# Bonus: Supervised Fine-Tuning Gemma 4 on a Sudoku Task

### An optional extra for the AMD Radeon GRPO workshop, built with [Unsloth](https://github.com/unslothai/unsloth)

In the main notebook you taught Gemma 4 with **reinforcement learning**: no answers, only a score. Here you try the other approach from Part 0, **supervised fine-tuning (SFT)**: you show the model correct answers and it learns to imitate them. You'll start with a toy version that runs on the CPU, then do it for real on Gemma.

**The task.** Puzzle sites and datasets store a Sudoku as an 81-character string, read row by row, with `.` or `0` for blanks. The GRPO notebook's `strategy(board, initial)` takes a 9x9 list of lists instead. You'll fine-tune the model to do that conversion. The base model usually writes a parser function instead of the board, or miscounts the digits; after SFT it outputs the board directly.

This suits SFT because correct answers are cheap to produce: a script wrote all of them.

---

### Prerequisites

This notebook assumes you've read Part 0 of the GRPO notebook, which covers LoRA and the difference between SFT and RL. It was validated in this environment:

- **Hardware:** 1 × AMD Radeon™ GPU (gfx1100) with 48 GB of memory. GPUs required: 1.
- **GPU memory:** about 14 GB at peak. You can leave the GRPO notebook's kernel running; together they use about 24 GB.
- **Software:** ROCm 7.2.4, PyTorch 2.10.0, Unsloth 2026.6.9, TRL 0.24.0 and Transformers 5.5.0, Python 3.12, as shipped in the workshop image (`unsloth_llama.cpp`).
- **Model:** [Gemma 4 E2B Instruct](https://huggingface.co/unsloth/gemma-4-E2B-it), the same base as the GRPO notebook. If your image has it under `/models`, it loads from there; otherwise it downloads from Hugging Face on first use.
- **Dataset:** `data/sudoku_sft_train.jsonl` (1,000 examples) and `data/sudoku_sft_test.jsonl` (50), in this repository. `data/make_sudoku_sft.py` generated them from a fixed seed and can regenerate them.

> Expected running time: about 10 minutes, most of it in the training cell.

## A toy SFT loop

Before you touch Gemma, see SFT on a model small enough to read in full. It's the same toy as the GRPO loop in the main notebook's Part 0: four numbers (logits) over four possible answers to *What is 2 + 2?*: `"3"`, `"4"`, `"5"` and `"four"`. This time you give it the correct answer, the label `"4"`. Each step computes the cross-entropy loss, which is low when the model puts its probability on the label, and nudges the logits to lower it.

Run the cell; it takes a few seconds on the CPU. Expected output: a table in which the loss falls steadily and `P(4)` rises from 25% toward 100%. `"four"` drops exactly as fast as `"3"` and `"5"`, because SFT only knows the label and every other answer counts as wrong; in the GRPO toy, `"four"` earned partial reward. Unlike in GRPO, the loss measures progress, as you'll also see when you train Gemma below.

In [ ]:
import torch
import torch.nn.functional as F

# The same toy "model" as the main notebook's GRPO loop: one logit per possible answer to "What is 2 + 2?".
answers = ["3", "4", "5", "four"]
label = torch.tensor([answers.index("4")])  # SFT needs the correct answer up front

logits = torch.zeros(len(answers), requires_grad = True)  # untrained: every answer equally likely
optimizer = torch.optim.SGD([logits], lr = 0.5)

print(f"{'step':>4} | {'loss':>5} |" + "".join(f"{'P(' + a + ')':>8}" for a in answers))
for step in range(1, 31):
    loss = F.cross_entropy(logits.unsqueeze(0), label)  # low when the model puts its probability on the label
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step <= 3 or step % 5 == 0:
        new_probs = torch.softmax(logits, dim = -1).tolist()
        print(f"{step:>4} | {loss.item():5.2f} |" + "".join(f"{p:>8.0%}" for p in new_probs))

## Setup

Gemma 4 needs `timm` for its vision and audio components, even though you only use text here. Unsloth and TRL are already installed in the workshop image.

In [ ]:
%%capture
!pip install --no-deps --upgrade timm

## Load the model

`FastModel.from_pretrained` loads Gemma 4 E2B Instruct in 16-bit, as in the GRPO notebook. The examples are short, so a 1,024-token context is enough.

Expected output: Unsloth's banner with the GPU name and package versions.

In [ ]:
from unsloth import FastModel
import os
import torch

# The workshop image ships the model under /models; elsewhere, download it from Hugging Face.
model_name = "/models/unsloth/gemma-4-E2B-it"
if not os.path.isdir(model_name):
    model_name = "unsloth/gemma-4-E2B-it"

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 1024,
    load_in_4bit = False,
)

## The dataset

Each line of the JSONL files is one example: a `messages` chat in which the user gives a puzzle string and the assistant answers with the board, plus the same board in a `board` field for scoring. The prompts use four different wordings and both `.` and `0` for blanks, so the model learns the task rather than one phrasing.

Expected output: the dataset size and one example.

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files = "data/sudoku_sft_train.jsonl", split = "train")
test_dataset = load_dataset("json", data_files = "data/sudoku_sft_test.jsonl", split = "train")
print(train_dataset)
for message in train_dataset[0]["messages"]:
    print(f"--- {message['role']} ---\n{message['content']}")

## The untrained baseline

Before training, see how the base model does. `evaluate` asks it to convert 20 held-out puzzles and counts a puzzle as correct only if all 81 cells match. `parse_board` pulls the `board = [...]` list out of each response. The 20 prompts go through as one batch, padded on the left.

Expected output: few or no exact boards. The printed example usually shows the base model writing a parser function, or a board with digits in the wrong places.

In [ ]:
import ast, re

def parse_board(response):
    match = re.search(r"board\s*=\s*(\[.*\])", response, re.S)
    try:
        return ast.literal_eval(match.group(1)) if match else None
    except (SyntaxError, ValueError):
        return None

def evaluate(n = 20, show = 1):
    examples = test_dataset.select(range(n))
    texts = [tokenizer.apply_chat_template(ex["messages"][:1], tokenize = False, add_generation_prompt = True)
             for ex in examples]
    tokenizer.tokenizer.padding_side = "left"  # Batched generation pads on the left
    inputs = tokenizer(text = texts, add_special_tokens = False, padding = True, return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 320, do_sample = False, use_cache = True)
    correct = 0
    for i, ex in enumerate(examples):
        response = tokenizer.decode(outputs[i][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
        correct += parse_board(response) == ex["board"]
        if i < show:
            print(f"--- prompt ---\n{ex['messages'][0]['content']}\n--- model ---\n{response}\n")
    print(f"Exact boards: {correct}/{n}")

evaluate()

## Attach LoRA adapters

The adapters go on the language layers only; the vision and audio towers stay frozen. `r = 32` with `lora_alpha = 32` gives enough capacity for a formatting task like this one.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## Set up the trainer

Each chat is rendered with Gemma's chat template into a single `text` field. Keep the leading `<bos>` token: a fine-tune trained without it tends to ramble. `train_on_responses_only` then masks the user's turn, so the loss covers only the assistant's answer and the model isn't trained to reproduce puzzle strings.

The main settings: a batch of 4 with 2 accumulation steps (8 examples per step), and `max_steps = 120`, about one pass over the 1,000 examples. No checkpoints are saved during training; you save the adapters yourself at the end.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

train_dataset = train_dataset.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize = False)})

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer.tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        max_steps = 120,  # About one pass over the 1,000 examples
        warmup_steps = 5,
        learning_rate = 2e-4,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        logging_steps = 10,
        save_strategy = "no",
        seed = 3407,
        report_to = "none",
        output_dir = "outputs/sft",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n")

## Train

Here the loss measures progress: it's how well the model predicts the correct answer. Watch it fall from about 0.2 in the first log lines toward 0.01.

In [ ]:
trainer.train()

## Test the trained model

Run the same 20 puzzles again.

Expected output: most of them exactly right, in the `board = [...]` format the dataset taught. The remaining mistakes are usually a miscounted run of blanks, which shifts every digit after it.

In [ ]:
evaluate()

## Save the adapters

You only need to save the LoRA adapters, to `sft_saved_lora`. The export options at the end of the GRPO notebook (merged weights, GGUF) work here too.

In [ ]:
model.save_pretrained("sft_saved_lora")
tokenizer.save_pretrained("sft_saved_lora")

## SFT versus GRPO

| | SFT (this notebook) | GRPO (main notebook) |
| --- | --- | --- |
| Training data | Prompts with correct answers | Prompts only |
| Learning signal | Match the answer, token by token | Reward from running the output |
| Good when | Correct answers are cheap to produce | Checking an answer is easier than writing one |

The two are often combined: a short SFT run teaches the output format, then RL improves the content.

## Exercises

1. Train on fewer examples (`train_dataset.select(range(100))`) and see how accuracy changes.
2. Lower `max_steps` to 30, or raise it to 250.
3. Add your own prompt wordings to `PROMPTS` in `data/make_sudoku_sft.py`, regenerate the data, and retrain.
4. Check that the model is still a generalist by asking it an unrelated question, as the GRPO notebook's generalist check does.